# Soup R12: R11 fusions with eager tied storage

**Prepared candidate; no new GPU result or admission claim.** It retains the R11 sparse cached native head, fused native CE adjoint, NF4 forward/backward cache and direct linear path, compact MLP activations, within-batch MLP padding compaction, variable-length attention, rotary, RMSNorm, LoRA dropout and epilogue, and packed metrics. The credited starter SwiGLU remains excluded from original-kernel credit.

Two fixed changes: NF4 cache capacity is 1 GiB (historical R11 used 3 GiB), and the native FP16 tied embedding/head table stays in that storage during source-pinned PEFT preparation. Only its redundant widening allocation is skipped; other parameter freezing, casts, hooks, checkpoint and cache behavior remain native. Embedding output stays FP32; no nominal weight value changes.

The original R11 run measured 63.193 seconds / 9,099 MiB / 12.39% original-kernel share. Those measurements belong to the old candidate and are not predictions for this revision. The 1,359 MiB target remains unproven: a 1 GiB decoded-weight cache alone exceeds the memory headroom left by the model.

Local validation: 26 structural/differential CPU cases and source compilation passed. This environment has no Torch or GPU. The inherited physical T4 readiness gate, new no-table-upcast test, 79 Soup tests, full fixed training and public evaluation still must run. The measured harness command is unchanged except an explicit 80-second per-training cap, with `--only solution`.


In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')
# Add notebook cells here to replace solution/__init__.py and your kernels.
# The prepared starter is retained when you run this template unchanged.


In [ ]:
import base64, hashlib, io, stat, zipfile
from pathlib import PurePosixPath

# No imports of Torch/CUDA here. The training subprocess imports solution.
FUSION_SOLUTION_SHA256 = '74a24d99e084e938fb96fdb41dad69b745e71492de29f898a7a5385a80572bc8'
FUSION_ARCHIVE_SHA256 = '53ce5d21feb2a3b28bbd8a830c2b65f648ce0a91f7c6c5624e6d73407f9e5ad0'
FUSION_FILES = {'solution/__init__.py': '493dc7b1b034632ddc8df2edfc6b8dde192760b992b15a931ff001183bb02b0c', 'solution/activation_storage.py': '95f66325e0361ced6e0c56ad06a027c66ac8d371969dc44a5840f746dfd21ca8', 'solution/cached_ce_backward.py': 'aab6f22ca12ccc343771726cf51f434b0fc550da0c262c86a78faf7001d62880', 'solution/cached_head.py': '0f49d819a7bc546c1a3252bfd13ab96f421cd2e160603942d014512197d488b2', 'solution/frozen_storage.py': '1ff7e662d3e4fe91b5c883bbe68b51c7ed5857dfbfa8ee43b79f19857d6f3efb', 'solution/head_fused_t4.py': 'ac75f93ce5b46eeb564afd8c74409cbbeda36804c0399d639c159bf8779b14ce', 'solution/head_matmul.py': 'c834bc001e3c6f9449ef4f5cd60d34aa961262e27c81e98e9a47186af5ec7e39', 'solution/head_matmul_t4.py': 'b29ed64e0d1537f6682d2b75d5a979c8269030a306aa8f1dee263bb9e3be3134', 'solution/integration.py': '4d45065b707b4c461b2968b72b7115a918845c95c7a66869253b6c9369def5d9', 'solution/lora_dropout.py': 'c6f4df8cc95eac7a76b36b0a84c0372b2678192cabe7d4227302378587bdd277', 'solution/lora_epilogue.py': '4fbbd8697ce95283458e81b6afa7aca581b9604eef7ae5f4565d7b38d2ac6f2f', 'solution/metric_transfer.py': '7187453ba1db76dbfa9cca4db3311cedf571dba2489e858a15b640ec41edf94a', 'solution/mlp_padding.py': 'c45fb840190aaf71132a9565cc132b4b5bdc117fd4ba4b85d9627b0094b794b9', 'solution/native_head_stats.py': '7bf0d63f9bd4e7b520ee0673df4cf12924f4e6a22562eee69faf4a93fec7f0cd', 'solution/nf4.py': '9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a', 'solution/nf4_cached_linear.py': '98bd9524243c564b49406c1e9f05bcb209764e8cc8ddeaa1d55befef99c036c3', 'solution/nf4_forward_cache.py': '10746bd3fe5e811f2ef05b957e08273dcea187bbbef56eeca69198e123661ea3', 'solution/nf4_forward_cache_patch.py': '8b1c4a0f6e14bf8cd9e554af2ece3ddafc7fe2777600f2177237b09f57b74d6a', 'solution/prepare_retained.py': '19e1f2b1b5885db71d152e6ecc0d4a9d5d4bb767b66ea4cfe3ea1e41ba2eae51', 'solution/rmsnorm.py': 'bf2c73ff515745e0d071890e011f1fd926fcb1b3415d1c978b56bf51bd47ed5a', 'solution/rotary.py': '8e8ef322d70eba6ac898a040a14d74b9dc8e276b3f93bd033414eeaf927b5429', 'solution/settings.py': 'e0976e04c16065393570124a36bd62adc0bdfb8fa7653d6224926fbad8c41df6', 'solution/shared_attention_mask.py': '93b6f6c2704685075fb14ac29901445350e0cf1c89a453ee1ef6e9500bd745d7', 'solution/sparse_cached_head.py': '73292b54b259236f47cfb1bfcdd2e2e23311fb1825d3c1e706543a04327c18bf', 'solution/streamed_head.py': '65ad6cada653b0d03d70375570b132c69f47a52ee8ff65ef0cb11042a63c9f19', 'solution/swiglu.py': 'e665910faeca6091c958ed7a191604484a44216f51d6fa069f5f48220e7472e6', 'solution/test_activation_storage.py': '4a4b8b9819c26745fb10ef8228119981fec75012b140844c43552f0ac42291cf', 'solution/test_cached_ce_backward.py': '5ac688fb4b2f63ad426ac37412fe87165debd28bab076814a94d3e4d8abeac55', 'solution/test_cached_ce_integration.py': '1c06a6605564bc0a7c9dad462f5b222cb993ee72585f8b37acbce430f42fb581', 'solution/test_cached_head.py': '01273dbd8b6c030c0210566decc727b827f177350dbcbb3f0764b277bcf56023', 'solution/test_frozen_storage.py': '9bbba62ed20bc54871a5a6f76102f5c8010053492c0a57024848b1b04c86f2af', 'solution/test_fusion_cuda.py': '4a699fc598234033048c8463300120317a6dc243efc29d1288b520322684b71b', 'solution/test_head_fused_t4.py': '335d684aa6fdc063d478265209e6ee05fcdf32fa7babe8d3e6c24644679d512d', 'solution/test_head_matmul.py': '1ae409f55282fbd96d3c78845ab22cded87bf453149fbecbbfe7e3f62ec4c9bd', 'solution/test_head_matmul_t4.py': '75931a45b9623e2ceb458904166218d31ffc68ccb08f129da5691cbdc961467d', 'solution/test_integration.py': '99bfd6cc33f692e0080d5b21c6754ba4c04ac116680d787e6f8e2f3d022b6b84', 'solution/test_kernels.py': 'a45b842a4f22c1aefe404baeff1ac26511253452ecb0f633c985fbc9df8b252a', 'solution/test_lora_dropout.py': '13d096500853e214811ee54dafc8c4c21c74f84e6e5ca0fe6b9a7599d01b055a', 'solution/test_lora_epilogue.py': 'c7394c871558fe12ab684aca5e5872ecd18763467e2ace460ff9016c7efd6421', 'solution/test_lora_joint_cuda.py': '5478c7012237ea429b23e76dacbf36ccf7df7e7c64ae9adf4ad77b8a5b54e2e4', 'solution/test_metric_transfer.py': '2540b1ca2a0b3ef42297bead8ee70fa3f0d2d4c71565ba11bc05b08a2eff9c19', 'solution/test_mlp_padding.py': '9629e2bee6fa1f135ab68ae4817d3702cce07728c2ab13d1162ed34b7f81397f', 'solution/test_native_head_stats.py': '17432c61dcb7eaf3d393b0ec246ae3af1678659fbedd9f622ef83fa43b075c45', 'solution/test_nf4.py': '2ee2f46a3011ab76c61675bef3b3e1ce40e0964d1737c92c36b71e8fd05c9a4f', 'solution/test_nf4_backward_cache.py': 'dc8433388ec96b4e40be0ce02025bc548d0480da1c1bdedfac51cc95d6942297', 'solution/test_nf4_cached_linear.py': '6cf204f438d12b2f6bfdd8ecbc93cf29c208a07e0cc95e7ce980337addefbcf0', 'solution/test_nf4_forward_cache.py': '4accdbc1b2d50b87d780f33da528b39221aca355337ab19cc7ba8db884b1a8a8', 'solution/test_rmsnorm.py': 'c14a0ac58ec70ffdc392c0a0d4cba3535e36754095130d79329967326110cc1a', 'solution/test_rotary.py': 'a8659bbb8455c6f20868672a7a41e4c96c67b5a066cd443addd0067acf4faf37', 'solution/test_shared_attention_mask.py': '7e9a36143655f96b8b95024643833f57addf4557ca8c759e0f9243a36bedeaa3', 'solution/test_soup_fusion_factory.py': '63b7b59d9a1d03991861e5afb72fbd6b36bb5370e104f176c9c540b84ffd81a1', 'solution/test_sparse_cached_head.py': '0023bb4b598513f2139106cd638c1c04d42062119b986630dcfb12c032303979', 'solution/test_sparse_cached_integration.py': '236ab4225831ed2e0519a70f86944054348abce4540705ba07b2e6060f0956d1', 'solution/test_streamed_head.py': '5ebbca158118591e39fca8b70ecf9f2485b287f07715573ab1c271acf21775d4', 'solution/test_varlen_attention.py': '93320a7e2388e096f0ea271bf27a5fae838aaa977a4ff3fb88918f98ad041d45', 'solution/test_varlen_attention_patch.py': 'd130a2cd390b934b794e7abaefe746b6399871dd581fc4fd4444d66080219417', 'solution/varlen_attention.py': '59b62e7aa1e5555a27309324563d77b9573c3b6d5257b57291a4cec9d1523f7b', 'solution/varlen_attention_patch.py': 'f012997c38bcc34a1fd56572c5aa877cecf23134c3cfb33ed9638e25c8f531fe', 'tools/run_t4_readiness.py': '3f241470789cbdceffc4866cc593f84a4d5044b60396335d2b9e04ae647bd0db'}
FUSION_PROTECTED = {'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821'}
FUSION_PAYLOAD = '...'
repo = Path(os.environ["SOUP_REPO"]).resolve(strict=True)
results = Path(os.environ["SOUP_RESULTS"])
results.mkdir(parents=True, exist_ok=True)
for name, expected in FUSION_PROTECTED.items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == expected, name
payload = base64.b64decode(FUSION_PAYLOAD, validate=True)
assert hashlib.sha256(payload).hexdigest() == FUSION_ARCHIVE_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == set(FUSION_FILES)
    assert len(archive.namelist()) == len(FUSION_FILES)
    for item in archive.infolist():
        name = PurePosixPath(item.filename)
        assert not name.is_absolute() and ".." not in name.parts
        assert name.parts[0] == "solution" or str(name) in ('tools/run_t4_readiness.py',)
        assert stat.S_ISREG(item.external_attr >> 16)
        raw = archive.read(item)
        assert hashlib.sha256(raw).hexdigest() == FUSION_FILES[str(name)], str(name)
        target = repo.joinpath(*name.parts)
        assert target.resolve().is_relative_to(repo) and not target.is_symlink()
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)

digest = hashlib.sha256()
for path in sorted((repo / "solution").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        digest.update(str(path.relative_to(repo / "solution")).encode())
        digest.update(path.read_bytes())
assert digest.hexdigest() == FUSION_SOLUTION_SHA256, "Unexpected extra or changed solution source"
for name, expected in FUSION_PROTECTED.items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == expected, name
(results / "fusion_source_manifest.json").write_text(json.dumps({
    "solution_sha256": FUSION_SOLUTION_SHA256,
    "archive_sha256": FUSION_ARCHIVE_SHA256,
    "files": FUSION_FILES,
    "protected_sources_unchanged": True,
    "service_template_source_commit": '3a4953f76288969bb91956f114eda5b7e541f104',
    "target_seconds": 45,
    "performance_status": "not_measured_by_install_cell"
}, indent=2) + "\n")
print("Installed fused candidate:", FUSION_SOLUTION_SHA256)
print("All optimization source is on disk under", repo / "solution")


In [ ]:
# Selected-source tests run in child processes, then exit before measurement.
# The helper uses a disposable test compiler cache, removed before this returns.
# Bounded reports are copied to SOUP_RESULTS even when a readiness gate fails.
# It does not run training or modify the template's measurement command.
import tempfile, importlib.metadata

# The service lacks pytest. This source-only, RECORD-verified dependency bundle
# is visible only to test subprocesses; no installed packages or parent
# environment variables are changed. Licenses and distribution metadata remain.
OFFLINE_TEST_SHA256 = '9fbe4277a9fbb84d03e0f0c59a77b80d87cee53a0ff49b2dd3952e7fcdf9cbf9'
OFFLINE_TEST_PAYLOAD = (
    '...'
)
test_raw = base64.b64decode(OFFLINE_TEST_PAYLOAD, validate=True)
assert hashlib.sha256(test_raw).hexdigest() == OFFLINE_TEST_SHA256
test_dependencies = Path(tempfile.mkdtemp(prefix="fusion-test-deps-", dir="/kaggle/working"))
with zipfile.ZipFile(io.BytesIO(test_raw)) as test_archive:
    test_manifest = json.loads(test_archive.read("offline_test_manifest.json"))
    assert set(test_archive.namelist()) == set(test_manifest["files"]) | {"offline_test_manifest.json"}
    assert len(test_archive.infolist()) == len(test_manifest["files"]) + 1
    assert sum(item.file_size for item in test_archive.infolist()) <= 10 * 1024**2
    for item in test_archive.infolist():
        member = PurePosixPath(item.filename)
        assert not member.is_absolute() and ".." not in member.parts
        assert stat.S_ISREG(item.external_attr >> 16)
        raw = test_archive.read(item)
        if item.filename != "offline_test_manifest.json":
            assert hashlib.sha256(raw).hexdigest() == test_manifest["files"][item.filename]
        target = test_dependencies.joinpath(*member.parts)
        assert target.resolve().is_relative_to(test_dependencies)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = os.pathsep.join(filter(None, [str(test_dependencies), os.environ.get("PYTHONPATH", "")]))
test_environment["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
test_environment["PYTHONDONTWRITEBYTECODE"] = "1"
test_environment["SOUP_READINESS_TEST_DEPS"] = str(test_dependencies)
base_test_versions = {}
for test_name in ("pytest", "pluggy", "iniconfig", "pygments", "packaging"):
    try:
        base_test_versions[test_name] = importlib.metadata.version(test_name)
    except importlib.metadata.PackageNotFoundError:
        base_test_versions[test_name] = None
reused_service_test_packages = []
packaging_version = base_test_versions.get("packaging")
if packaging_version and packaging_version.split(".", 1)[0].isdigit() and int(packaging_version.split(".", 1)[0]) >= 24:
    # Keep Torch/Transformers on their existing packaging implementation.
    shutil.rmtree(test_dependencies / "packaging")
    for metadata_directory in test_dependencies.glob("packaging-*.dist-info"):
        shutil.rmtree(metadata_directory)
    reused_service_test_packages.append("packaging")
readiness_work = Path("/kaggle/working/fusion-readiness")
readiness_reports = results / "fusion_readiness"
readiness_reports.mkdir(parents=True, exist_ok=True)
copied, omitted, report_bytes = [], [], 0
try:
    # Optional component diagnostics execute before the same mandatory gate.
    # They never relax that gate or modify the measured training command.
    readiness_work.mkdir(parents=True, exist_ok=True)
    for helper_name in ():
        label = Path(helper_name).stem
        with tempfile.TemporaryDirectory(prefix="fusion-probe-cache-", dir="/kaggle/working") as probe_cache:
            probe_environment = dict(test_environment,
                TRITON_CACHE_DIR=str(Path(probe_cache) / "triton"),
                TORCHINDUCTOR_CACHE_DIR=str(Path(probe_cache) / "inductor"))
            probe_arguments = (["--run", "--output", str(readiness_work / (label + ".json"))]
                               if label in ("probe_head_numerics", "probe_attention_dispatch", "probe_ce_backward") else
                               ["--out", str(readiness_work / label)])
            with (readiness_work / (label + ".log")).open("w") as probe_log:
                probe_result = subprocess.run([
                    sys.executable, str(repo / helper_name), *probe_arguments
                ], cwd=repo, env=probe_environment, stdout=probe_log,
                   stderr=subprocess.STDOUT, timeout=600)
            if probe_result.returncode:
                raise RuntimeError("Physical component diagnostic failed: " + label)
    readiness = subprocess.run([
        sys.executable, str(repo / "tools/run_t4_readiness.py"),
        "--local-snapshot", "--output-dir", str(readiness_work)
    ], cwd=repo, env=test_environment)
finally:
    shutil.rmtree(test_dependencies)
    for path in sorted(readiness_work.rglob("*")):
        if not path.is_file() or path.is_symlink():
            continue
        relative, size = str(path.relative_to(readiness_work)), path.stat().st_size
        if (path.suffix not in (".json", ".jsonl", ".log", ".yaml")
                or size >= 4 * 1024**2 or report_bytes + size > 8 * 1024**2
                or len(copied) >= 20):
            omitted.append({"file": relative, "bytes": size})
            continue
        target = readiness_reports / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
        copied.append(relative)
        report_bytes += size
    (readiness_reports / "export_manifest.json").write_text(json.dumps({
        "copied_reports": copied, "omitted_reports": omitted,
        "copied_bytes": report_bytes, "readiness_workspace": str(readiness_work),
        "export_limit_bytes": 8 * 1024**2, "export_limit_files": 20,
        "test_dependencies_sha256": OFFLINE_TEST_SHA256,
        "test_dependencies": test_manifest["distributions"],
        "service_test_package_versions_before_injection": base_test_versions,
        "reused_service_test_packages": reused_service_test_packages,
        "test_dependencies_removed_before_training": not test_dependencies.exists(),
        "parent_environment_modified": False
    }, indent=2) + "\n")
if readiness.returncode:
    raise RuntimeError("Physical T4 readiness failed; inspect fusion_readiness reports before training")


In [ ]:
# Preserve the original export cell and also retain reports if training raises.
# This synchronous, failure-only hook does no work while training is running.
from IPython import get_ipython

def _fusion_export_failed_cell(execution):
    if execution.error_in_exec is None and execution.error_before_exec is None:
        return
    failed_run = globals().get("run")
    if not isinstance(failed_run, Path) or not failed_run.is_dir():
        return
    copied, errors = [], []
    try:
        for path in failed_run.rglob("*"):
            if (path.is_file() and not path.is_symlink()
                    and path.suffix in (".json", ".jsonl", ".log", ".yaml")
                    and path.stat().st_size < 4 * 1024**2):
                target = results / path.relative_to(failed_run)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(path, target)
                copied.append(str(path.relative_to(failed_run)))
    except Exception as error:
        errors.append({"error": str(error)})
    try:
        (results / "fusion_failed_cell_export.json").write_text(json.dumps({
            "trigger": "notebook_cell_error", "copied_reports": copied,
            "copy_errors": errors, "training_success_claimed": False
        }, indent=2) + "\n")
        print("Retained failed-cell reports:", len(copied), "copy errors:", len(errors))
    except Exception as error:
        print("Could not write failure-export summary:", type(error).__name__)

shell = get_ipython()
assert shell is not None, "Run this artifact as a notebook in the prepared service"
shell.events.register("post_run_cell", _fusion_export_failed_cell)


In [ ]:
# Run only the solution. The server reuses its saved matching public baseline.
run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '80', '--only', 'solution'], cwd=repo, check=True)


In [ ]:
# Export small reports and logs. Large adapter/checkpoint files remain in this job's VM workspace.
for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())
# This is a public-data screen. Notebook output does not establish official admission or score.
